In [3]:
# Install required libraries
!pip install nltk spacy seaborn
!python -m spacy download en_core_web_sm

# Imports
import nltk
from nltk.util import ngrams
from collections import defaultdict, Counter
import spacy
import seaborn as sns
import pandas as pd
import math
from sklearn.metrics import accuracy_score, precision_recall_fscore_support



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Task 1: Language Modeling (Bigram Model)**

**Step 1: Load & Preprocess Text**

In [10]:
nltk.download('punkt')

# Load dataset
from datasets import load_dataset

# Load Emotion dataset
dataset = load_dataset("emotion")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
train_data = dataset["train"]

**Task 1: Language Modeling — Bigram Next Word Prediction**

**Preprocess and Build Bigram Model**

In [13]:
import nltk
from nltk.util import ngrams
from collections import defaultdict, Counter

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
nltk.download("punkt_tab")

# Tokenize
tokenized = [nltk.word_tokenize(text.lower()) for text in train_data["text"]]

# Build bigram counts
bigram_counts = defaultdict(Counter)
for sentence in tokenized:
    for w1, w2 in ngrams(sentence, 2):
        bigram_counts[w1][w2] += 1

# Convert to probabilities
bigram_prob = {}
for w1, counter in bigram_counts.items():
    total = sum(counter.values())
    bigram_prob[w1] = {w2: count/total for w2, count in counter.items()}

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


**Predict Next Word Function**

In [16]:
def predict_next(word):
    if word in bigram_prob:
        return max(bigram_prob[word], key=bigram_prob[word].get)
    return None

print("Next word for 'i':", predict_next("i"))
print("Next word for 'feel':", predict_next("feel"))

Next word for 'i': feel
Next word for 'feel': like


**Task 2: Sequence Tagging with spaCy**

**POS Tagging and NER**

In [17]:
import spacy

nlp = spacy.load("en_core_web_sm")

example_sentence = "Barack Obama visited Nairobi before speaking at the conference."

doc = nlp(example_sentence)

print("POS Tags:")
for token in doc:
    print(token.text, "/", token.pos_)

print("\nNamed Entities:")
for ent in doc.ents:
    print(ent.text, "/", ent.label_)

POS Tags:
Barack / PROPN
Obama / PROPN
visited / VERB
Nairobi / PROPN
before / ADP
speaking / VERB
at / ADP
the / DET
conference / NOUN
. / PUNCT

Named Entities:
Barack Obama / PERSON
Nairobi / GPE


**Task 3: Evaluation**

**Language Modeling: Perplexity (Simple Estimate)**

In [18]:
import math

def perplexity(sentence):
    tokens = nltk.word_tokenize(sentence.lower())
    prob = 1
    N = len(tokens)
    for w1, w2 in ngrams(tokens, 2):
        prob *= bigram_prob.get(w1, {}).get(w2, 1e-6)
    return math.pow(1/prob, 1/N)

print("Perplexity:", perplexity("i feel happy"))

Perplexity: 12.33609472151389
